In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [5]:
!unzip -q data.zip -d data

In [6]:
!find data -maxdepth 2 -type f | head -20

data/data/batches.meta
data/data/data_batch_3
data/data/test_batch
data/data/cifar-10-python.tar.gz
data/data/data_batch_2
data/data/data_batch_1
data/data/data_batch_4


In [7]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [16:26<00:00, 173kB/s]


In [8]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

Build the **CNN**

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [10]:
model = CNN()

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Training the CNN

In [12]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3895886084612679
epoch=2/10 & loss=0.9651753407762483
epoch=3/10 & loss=0.7739942726652945
epoch=4/10 & loss=0.6428257099655278
epoch=5/10 & loss=0.5372102109291365
epoch=6/10 & loss=0.441803584785184
epoch=7/10 & loss=0.35664840441797396
epoch=8/10 & loss=0.27851389127466686
epoch=9/10 & loss=0.2150405360638257
epoch=10/10 & loss=0.17115984739416548


In [13]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.94999999999999
